# P2 - Grayscale, Resize, Median Filter, dan Ekualisasi Histogram

In [17]:
import os
import csv
import math
import math as _math
import cv2
import numpy as np
from tqdm import tqdm

## Import Library

Dilakukan import beberapa library yang diperlukan untuk mendukung proses preprocessing citra dan ekstraksi fitur tekstur menggunakan metode Gray Level Co-occurrence Matrix (GLCM). Library `os` digunakan untuk mengakses, membaca, dan mengelola struktur folder dataset citra sehingga program dapat melakukan pemrosesan gambar secara otomatis pada setiap kategori data. Library `csv` digunakan untuk menyimpan hasil ekstraksi fitur ke dalam file berformat CSV yang nantinya akan digunakan sebagai dataset masukan pada tahap klasifikasi. Library `math` yang diimpor dengan alias `_math` digunakan untuk melakukan perhitungan matematis, khususnya fungsi logaritma pada perhitungan fitur Entropy. Selanjutnya, library `cv2` (OpenCV) digunakan untuk membaca citra digital dan mengubahnya menjadi citra grayscale karena metode GLCM bekerja pada citra satu kanal intensitas. Library `numpy` digunakan untuk membuat dan memanipulasi matriks GLCM serta mendukung berbagai operasi numerik yang diperlukan selama proses preprocessing dan ekstraksi fitur. Terakhir, library `tqdm` digunakan untuk menampilkan progress bar selama proses ekstraksi berlangsung sehingga memudahkan pemantauan perkembangan pemrosesan seluruh citra dalam dataset.

In [18]:
PATH_INPUT = "Assets/"
KATEGORI = [
    "Normal",
    "kidneyStone"
]
OUTPUT_CSV  = "hasil_ekstraksi_percobaan2.csv"
GLCM_DISTANCE = 1
GLCM_ANGLES = [0,45,90,135]

## Konfigurasi Program

Dilakukan konfigurasi parameter utama yang akan digunakan selama proses preprocessing dan ekstraksi fitur GLCM. Variabel `PATH_INPUT` digunakan untuk menentukan lokasi folder dataset yang berisi citra ginjal normal dan citra batu ginjal. Variabel `KATEGORI` berisi daftar kelas yang digunakan dalam percobaan, yaitu `Normal` dan `kidneyStone`, sehingga program dapat membaca dan memproses citra berdasarkan kategori masing-masing. Variabel `OUTPUT_CSV` digunakan untuk menentukan nama file keluaran yang akan menyimpan seluruh hasil ekstraksi fitur tekstur dalam format CSV. Selanjutnya, parameter `GLCM_DISTANCE` ditetapkan bernilai 1 yang menunjukkan jarak antar piksel yang digunakan dalam pembentukan pasangan piksel pada matriks GLCM. Variabel `GLCM_ANGLES` berisi empat orientasi sudut, yaitu 0°, 45°, 90°, dan 135°. Pada skenario preprocessing kedua (P2), setiap citra terlebih dahulu diubah menjadi citra grayscale, kemudian dilakukan resize menjadi ukuran 256×256 piksel, dilanjutkan dengan median filter untuk mengurangi noise, serta histogram equalization untuk meningkatkan kontras citra sebelum dilakukan ekstraksi fitur GLCM.

In [19]:
def manual_clip(arr, a_min, a_max):
    if isinstance(arr, np.ndarray):
        flat = arr.flat
        return np.array([max(a_min, min(a_max, x)) for x in flat]).reshape(arr.shape)
    else:
        return max(a_min, min(a_max, arr))

## Fungsi Manual Clip

Pada tahap ini dibuat fungsi `manual_clip()` sebagai pengganti fungsi bawaan NumPy yaitu `np.clip()`. Fungsi ini digunakan untuk membatasi nilai agar tetap berada pada rentang tertentu yang telah ditentukan. Jika suatu nilai lebih kecil dari batas minimum maka nilai tersebut akan diubah menjadi batas minimum, sedangkan jika nilainya melebihi batas maksimum maka akan diubah menjadi batas maksimum. Pada project ini fungsi digunakan selama proses resize citra dan histogram equalization untuk memastikan nilai koordinat maupun intensitas piksel tetap berada dalam rentang yang valid.

In [20]:
def manual_arange(start, stop=None, step=1):
    if stop is None:
        stop = start
        start = 0
    length = int(math.ceil((stop - start) / step))
    return np.array([start + i*step for i in range(length)], dtype=np.float64)

## Fungsi Manual Arange

Fungsi `manual_arange()` dibuat sebagai pengganti fungsi `np.arange()` yang umumnya digunakan untuk membentuk deret angka dengan interval tertentu. Fungsi ini menghasilkan kumpulan nilai yang digunakan sebagai dasar perhitungan koordinat piksel pada proses resize citra. Dengan membuat fungsi secara manual, proses perhitungan dapat dilakukan tanpa bergantung pada fungsi otomatis NumPy sehingga seluruh tahapan preprocessing dapat dikendalikan secara lebih transparan.

In [21]:
def manual_bincount(x, minlength=0):
    if minlength == 0:
        minlength = int(max(x)) + 1 if x.size > 0 else 0
    hist = [0] * minlength
    for val in x.flat:
        hist[int(val)] += 1
    return np.array(hist, dtype=np.float64)

## Fungsi Manual Bincount

Fungsi `manual_bincount()` dibuat sebagai pengganti fungsi `np.bincount()`. Fungsi ini digunakan untuk menghitung jumlah kemunculan setiap nilai intensitas piksel pada citra grayscale. Hasil perhitungan berupa histogram frekuensi yang menunjukkan distribusi tingkat keabuan citra. Histogram tersebut kemudian digunakan sebagai dasar dalam proses histogram equalization untuk meningkatkan kontras citra.

In [22]:
def manual_cumsum(arr):
    res = np.zeros_like(arr)
    s = 0.0
    for i in range(arr.size):
        s += arr.flat[i]
        res.flat[i] = s
    return res

## Fungsi Manual Cumulative Sum

Fungsi `manual_cumsum()` dibuat sebagai pengganti fungsi `np.cumsum()`. Fungsi ini digunakan untuk menghitung jumlah kumulatif dari setiap elemen histogram sehingga menghasilkan nilai Cumulative Distribution Function (CDF). Nilai CDF digunakan pada proses histogram equalization untuk membangun tabel pemetaan intensitas piksel baru yang dapat meningkatkan distribusi kontras citra.

In [23]:
def manual_pad_reflect(img, pad):
    h, w = img.shape
    padded = np.zeros((h + 2*pad, w + 2*pad), dtype=img.dtype)
    padded[pad:pad+h, pad:pad+w] = img
    # vertikal
    for i in range(pad):
        padded[i, pad:pad+w] = img[pad - i - 1, :]
        padded[h + pad + i, pad:pad+w] = img[h - i - 1, :]
    # horizontal
    for i in range(pad):
        padded[:, i] = padded[:, 2*pad - i - 1]
        padded[:, w + pad + i] = padded[:, w + pad - i - 1]
    return padded

## Fungsi Manual Padding Refleksi

Pada tahap ini dibuat fungsi `manual_pad_reflect()` yang digunakan untuk menambahkan padding pada tepi citra menggunakan metode refleksi. Teknik ini dilakukan dengan menyalin nilai piksel di sekitar batas citra secara simetris sehingga area tambahan memiliki karakteristik yang mirip dengan citra asli. Padding refleksi diperlukan pada proses median filtering agar piksel yang berada di tepi citra tetap dapat diproses menggunakan kernel tanpa menyebabkan kehilangan informasi atau error indeks.

In [24]:
def resize_manual(img, ukuran_baru):
    tinggi_lama, lebar_lama = img.shape
    lebar_baru, tinggi_baru = ukuran_baru
    img_f = img.astype(np.float64)

    skala_x = lebar_lama / lebar_baru
    skala_y = tinggi_lama / tinggi_baru

    y_asal = manual_clip(
        (manual_arange(tinggi_baru) + 0.5) * skala_y - 0.5,
        0, tinggi_lama - 1
    )
    x_asal = manual_clip(
        (manual_arange(lebar_baru) + 0.5) * skala_x - 0.5,
        0, lebar_lama - 1
    )

    hasil = np.zeros((tinggi_baru, lebar_baru), dtype=np.float64)

    for i in range(tinggi_baru):
        y = y_asal[i]
        y0 = int(math.floor(y))
        y1 = min(y0 + 1, tinggi_lama - 1)
        wy = y - y0
        for j in range(lebar_baru):
            x = x_asal[j]
            x0 = int(math.floor(x))
            x1 = min(x0 + 1, lebar_lama - 1)
            wx = x - x0

            p00 = img_f[y0, x0]
            p01 = img_f[y0, x1]
            p10 = img_f[y1, x0]
            p11 = img_f[y1, x1]

            atas  = p00 * (1 - wx) + p01 * wx
            bawah = p10 * (1 - wx) + p11 * wx
            hasil[i, j] = atas * (1 - wy) + bawah * wy

    return manual_clip(hasil, 0, 255).astype(np.uint8)

## Resize Citra Secara Manual

Pada tahap ini dilakukan proses resize citra menggunakan metode interpolasi bilinear yang diimplementasikan secara manual tanpa menggunakan fungsi bawaan seperti `cv2.resize()`. Fungsi `resize_manual()` bertujuan mengubah ukuran citra menjadi 256×256 piksel sehingga seluruh data memiliki dimensi yang seragam sebelum dilakukan ekstraksi fitur tekstur. Proses resize diawali dengan menghitung faktor skala pada sumbu horizontal dan vertikal, kemudian menentukan koordinat piksel asal yang sesuai pada citra awal. Nilai intensitas piksel baru diperoleh menggunakan interpolasi bilinear yang memanfaatkan empat piksel tetangga terdekat sebagai dasar perhitungan. Metode ini menghasilkan perubahan ukuran citra yang lebih halus dibandingkan pendekatan nearest neighbor karena mempertimbangkan kontribusi beberapa piksel di sekitarnya.

In [25]:
def median_filter_manual(img, kernel_size=3):
    pad = kernel_size // 2
    img_pad = manual_pad_reflect(img.astype(np.float64), pad)
    tinggi, lebar = img.shape
    hasil = np.zeros((tinggi, lebar), dtype=np.float64)

    for i in range(tinggi):
        for j in range(lebar):
            # ambil window
            window = []
            for ki in range(kernel_size):
                for kj in range(kernel_size):
                    window.append(img_pad[i+ki, j+kj])
            # urutkan (gunakan sorted bawaan Python, bukan numpy)
            window_sorted = sorted(window)
            median = window_sorted[len(window)//2]
            hasil[i, j] = median

    return hasil.astype(np.uint8)

## Median Filter Manual

Pada tahap ini dilakukan proses reduksi noise menggunakan metode Median Filter berukuran kernel 3×3 yang diimplementasikan secara manual. Setiap piksel pada citra akan digantikan dengan nilai median dari piksel-piksel yang berada pada area kernel di sekitarnya. Metode median filtering dipilih karena efektif dalam menghilangkan noise impulsif tanpa menghilangkan tepi objek secara signifikan. Hasil proses ini berupa citra yang lebih halus dan memiliki gangguan noise yang lebih rendah sehingga dapat meningkatkan kualitas fitur tekstur yang diekstraksi pada tahap berikutnya.

In [26]:
def histogram_equalization_manual(img):
    tinggi, lebar = img.shape
    total_piksel = img.size

    # histogram manual
    hist = manual_bincount(img.ravel(), minlength=256)

    # CDF manual
    cdf = manual_cumsum(hist)

    # cari nilai cdf minimum yang bukan nol (untuk normalisasi standar)
    cdf_min = 0.0
    for nilai in cdf:
        if nilai > 0:
            cdf_min = nilai
            break

    # bangun tabel pemetaan (lookup table) 0-255
    lut = np.zeros(256, dtype=np.float64)
    for i in range(256):
        if total_piksel - cdf_min > 0:
            lut[i] = (cdf[i] - cdf_min) / (total_piksel - cdf_min) * 255.0
        else:
            lut[i] = 0.0

    # terapkan lookup table ke setiap piksel
    hasil = np.zeros((tinggi, lebar), dtype=np.float64)
    for i in range(tinggi):
        for j in range(lebar):
            hasil[i, j] = lut[img[i, j]]

    return manual_clip(hasil, 0, 255).astype(np.uint8)

## Histogram Equalization Manual

Pada tahap ini dilakukan peningkatan kualitas citra menggunakan metode Histogram Equalization yang diimplementasikan secara manual tanpa menggunakan fungsi bawaan OpenCV. Proses diawali dengan menghitung histogram tingkat keabuan citra, kemudian membentuk Cumulative Distribution Function (CDF) yang digunakan untuk membuat tabel pemetaan intensitas baru. Melalui proses ini distribusi intensitas piksel menjadi lebih merata sehingga kontras citra meningkat. Histogram equalization membantu memperjelas detail tekstur pada citra ginjal sehingga informasi yang diperoleh pada proses ekstraksi fitur GLCM menjadi lebih representatif.

In [27]:
def build_glcm_manual(img, distance, angle_deg, levels=256):
    h, w = img.shape
    glcm = np.zeros((levels, levels), dtype=np.float64)

    if angle_deg == 0:
        di, dj = 0, distance
    elif angle_deg == 45:
        di, dj = -distance, distance
    elif angle_deg == 90:
        di, dj = -distance, 0
    elif angle_deg == 135:
        di, dj = -distance, -distance

    for i in range(h):
        for j in range(w):
            ni = i + di
            nj = j + dj
            if 0 <= ni < h and 0 <= nj < w:
                glcm[int(img[i, j]), int(img[ni, nj])] += 1

    glcm = glcm + glcm.T
    total = glcm.sum()
    if total > 0:
        glcm /= total
    return glcm

## Pembentukan Matriks GLCM

Pada tahap ini dilakukan pembentukan matriks GLCM secara manual tanpa menggunakan fungsi bawaan seperti `graycomatrix()`. Fungsi ini menghitung frekuensi kemunculan pasangan nilai intensitas piksel berdasarkan jarak dan orientasi sudut tertentu, kemudian membentuk matriks probabilitas yang digunakan sebagai dasar perhitungan fitur tekstur. Matriks GLCM dibentuk pada empat orientasi sudut yaitu 0°, 45°, 90°, dan 135° dengan jarak antar piksel sebesar satu piksel. Setelah seluruh pasangan piksel dihitung, matriks dibuat simetris dan dinormalisasi sehingga total probabilitas bernilai satu.

In [28]:
def ekstrak_fitur_glcm(glcm, levels=256):
    contrast = homogeneity = dissimilarity = entropy = asm = correlation = 0.0

    # Mean untuk korelasi
    mu_i = mu_j = 0.0
    for i in range(levels):
        for j in range(levels):
            mu_i += i * glcm[i, j]
            mu_j += j * glcm[i, j]

    # Standar deviasi untuk korelasi
    sigma_i = sigma_j = 0.0
    for i in range(levels):
        for j in range(levels):
            sigma_i += (i - mu_i) ** 2 * glcm[i, j]
            sigma_j += (j - mu_j) ** 2 * glcm[i, j]
    sigma_i = sigma_i ** 0.5
    sigma_j = sigma_j ** 0.5

    for i in range(levels):
        for j in range(levels):
            p = glcm[i, j]
            if p == 0:
                continue
            diff = i - j
            contrast      += (diff ** 2) * p
            homogeneity   += p / (1.0 + diff ** 2)
            dissimilarity += abs(diff) * p
            entropy       -= p * _math.log(p, 2)
            asm           += p ** 2
            if sigma_i > 0 and sigma_j > 0:
                correlation += ((i - mu_i) * (j - mu_j) * p) / (sigma_i * sigma_j)

    energy = asm ** 0.5
    return contrast, homogeneity, dissimilarity, entropy, asm, energy, correlation

## Ekstraksi Fitur Tekstur GLCM

Pada tahap ini dilakukan perhitungan tujuh fitur tekstur utama yang berasal dari matriks GLCM, yaitu Contrast, Homogeneity, Dissimilarity, Entropy, ASM, Energy, dan Correlation. Seluruh fitur dihitung secara manual berdasarkan rumus matematis masing-masing sehingga karakteristik tekstur citra dapat direpresentasikan secara kuantitatif untuk digunakan pada tahap klasifikasi. Fitur-fitur tersebut digunakan untuk menggambarkan pola hubungan antar piksel dan tingkat keragaman tekstur yang terdapat pada citra ginjal.

In [29]:
def proses_glcm_satu_gambar(img, distance=1, angles=[0, 45, 90, 135]):
    row = {}
    for angle in angles:
        glcm = build_glcm_manual(img, distance, angle)
        c, h, d, e, a, en, cor = ekstrak_fitur_glcm(glcm)
        row[f"Contrast{angle}"]      = c
        row[f"Homogeneity{angle}"]   = h
        row[f"Dissimilarity{angle}"] = d
        row[f"Entropy{angle}"]       = e
        row[f"ASM{angle}"]           = a
        row[f"Energy{angle}"]        = en
        row[f"Correlation{angle}"]   = cor
    return row

## Proses Ekstraksi Fitur GLCM pada Satu Citra

Pada tahap ini dilakukan integrasi proses pembentukan matriks GLCM dan ekstraksi fitur tekstur untuk satu citra. Matriks GLCM dibentuk pada empat orientasi sudut yaitu 0°, 45°, 90°, dan 135°, kemudian dari masing-masing sudut dihitung tujuh fitur tekstur. Hasil akhir berupa 28 fitur yang digunakan sebagai representasi tekstur dari satu citra. Seluruh fitur tersebut disimpan dalam bentuk dictionary sehingga dapat digabungkan dengan data citra lainnya pada tahap pembentukan dataset.

In [30]:
CSV_HEADER = ["Filename", "Label"]

for _angle in GLCM_ANGLES:
    for _fitur in [
        "Contrast",
        "Homogeneity",
        "Dissimilarity",
        "Entropy",
        "ASM",
        "Energy",
        "Correlation"
    ]:
        CSV_HEADER.append(f"{_fitur}{_angle}")

## Pembuatan Header Dataset Fitur

Pada tahap ini dilakukan pembuatan struktur header yang akan digunakan sebagai nama kolom pada file CSV hasil ekstraksi fitur. Header terdiri dari kolom Filename, Label, serta 28 fitur tekstur hasil perhitungan GLCM dari empat orientasi sudut yang berbeda. Struktur header ini memastikan setiap fitur tersimpan pada kolom yang sesuai sehingga memudahkan proses analisis dan klasifikasi pada tahap selanjutnya.

## Proses Ekstraksi Fitur Seluruh Dataset

Pada tahap ini dilakukan proses ekstraksi fitur untuk seluruh citra dalam dataset. Setiap citra dibaca dalam format grayscale, kemudian dilakukan resize menjadi 256×256 piksel menggunakan interpolasi bilinear. Setelah itu diterapkan median filter berukuran 3×3 untuk mengurangi noise pada citra dan dilanjutkan dengan histogram equalization untuk meningkatkan kontras citra. Citra hasil preprocessing kemudian digunakan untuk membentuk matriks GLCM pada empat orientasi sudut yaitu 0°, 45°, 90°, dan 135°. Selanjutnya dihitung tujuh fitur tekstur untuk setiap sudut sehingga menghasilkan total 28 fitur per citra. Hasil ekstraksi kemudian disimpan ke dalam struktur data yang akan digunakan untuk membentuk dataset klasifikasi.

In [ ]:
print("\nMemulai ekstraksi fitur GLCM ...")
semua_baris = []

for label in KATEGORI:
    folder = os.path.join(PATH_INPUT, label)
    if not os.path.exists(folder):
        print(f"Folder tidak ditemukan: {folder}")
        continue
    for nama_file in tqdm(sorted(os.listdir(folder)), desc=label):
        if not nama_file.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        jalur = os.path.join(folder, nama_file)
        img = cv2.imread(jalur, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        img = resize_manual(img,(256,256))
        img = median_filter_manual(img, kernel_size=3)
        img = histogram_equalization_manual(img)
        fitur_row = proses_glcm_satu_gambar(img, GLCM_DISTANCE, GLCM_ANGLES)
        baris = {"Filename": nama_file,"Label": label}
        baris.update(fitur_row)
        semua_baris.append(baris)


Memulai ekstraksi fitur GLCM ...


kidneyStone: 100%|██████████| 100/100 [01:39<00:00,  1.00it/s]


## Penyimpanan Hasil Ekstraksi Fitur ke File CSV

Pada tahap terakhir dilakukan penyimpanan seluruh hasil ekstraksi fitur ke dalam file CSV. Data yang telah dikumpulkan dari seluruh citra dituliskan menggunakan `csv.DictWriter()` sesuai struktur kolom yang telah ditentukan sebelumnya.

In [ ]:
with open(OUTPUT_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=CSV_HEADER)
    writer.writeheader()
    writer.writerows(semua_baris)

print(f"{len(semua_baris)} data tersimpan ke: {OUTPUT_CSV}")

print(f"\n Selesai! CSV tersimpan di: {OUTPUT_CSV}")

  ✅ 200 data tersimpan ke: hasil_ekstraksi_percobaan2.csv

🎉 Selesai! CSV tersimpan di: hasil_ekstraksi_percobaan2.csv
